In [1]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D12 — Our World in Data CO2 Dataset
# ============================================================

from google.colab import files
from pathlib import Path

import csv
import hashlib
import json
import platform
import re
import sys

import pandas as pd


In [2]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D12"

DOCUMENT_NAME = (
    "Our World in Data — Annual CO2 emissions time series"
)

SOURCE_FORMAT = "CSV"

EXPECTED_COLUMNS = [
    "Year",
    "Annual CO2 emissions"
]

REFERENCE_YEARS = [
    1750,
    1800,
    1850,
    1900,
    1950,
    1960,
    1970,
    1980,
    1990,
    2000,
    2010,
    2011,
    2012,
    2013,
    2014,
    2015,
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
]


REFERENCE_CATEGORY = (
    "Environmental time-series"
)

REFERENCE_UNIT = None

EXPECTED_REFERENCE_RECORD_COUNT = len(
    REFERENCE_YEARS
)

REFERENCE_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

OUTPUT_DIR = Path("outputs_D12_stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR / "D12_document_characterisation.json"
)

DATASET_STATISTICS_PATH = (
    OUTPUT_DIR / "D12_dataset_statistics.json"
)

INTEGRITY_REPORT_PATH = (
    OUTPUT_DIR / "D12_integrity_report.json"
)

REFERENCE_VALUES_PATH = (
    OUTPUT_DIR / "D12_reference_values.csv"
)

COLUMN_PROFILE_PATH = (
    OUTPUT_DIR / "D12_column_profile.csv"
)

REFERENCE_VALUES_JSON_PATH = (
    OUTPUT_DIR / "D12_reference_values.json"
)

REFERENCE_SCHEMA_PATH = (
    OUTPUT_DIR / "D12_reference_schema.json"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR / "D12_extraction_schema.json"
)

EXTRACTION_TASK_PATH = (
    OUTPUT_DIR / "D12_extraction_task.txt"
)

REFERENCE_SUMMARY_PATH = (
    OUTPUT_DIR / "D12_reference_summary.json"
)

REFERENCE_METADATA_PATH = (
    OUTPUT_DIR / "D12_reference_metadata.json"
)

REFERENCE_INTEGRITY_PATH = (
    OUTPUT_DIR / "D12_reference_integrity.json"
)

INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D12_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D12_dimension_assessment.csv"
)

QUALITY_EVIDENCE_PATH = (
    OUTPUT_DIR / "D12_quality_evidence.json"
)

print("Document:", DOCUMENT_ID)
print("Expected reference records:", EXPECTED_REFERENCE_RECORD_COUNT)
print("Output directory:", OUTPUT_DIR)


Document: D12
Expected reference records: 25
Output directory: outputs_D12_stage1


In [3]:
# ============================================================
# 2. Upload original CSV
# ============================================================

print("Upload the original D12 CSV file.")

uploaded = files.upload()

csv_paths = [
    Path(filename)
    for filename in uploaded
    if filename.lower().endswith(".csv")
]

if len(csv_paths) != 1:
    raise ValueError(
        "Upload exactly one CSV source file."
    )

SOURCE_PATH = csv_paths[0]

print("Source file:", SOURCE_PATH.name)
print("Source size:", f"{SOURCE_PATH.stat().st_size:,} bytes")


Upload the original D12 CSV file.


Saving D12 - ourworldindataCO2.csv to D12 - ourworldindataCO2.csv
Source file: D12 - ourworldindataCO2.csv
Source size: 4,537 bytes


In [4]:
# ============================================================
# 3. File hashing and delimiter detection
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def detect_delimiter(path):
    sample = path.read_text(
        encoding="utf-8-sig",
        errors="replace"
    )[:10000]

    try:
        dialect = csv.Sniffer().sniff(
            sample,
            delimiters=[",", ";", "\t", "|"]
        )
        return dialect.delimiter

    except csv.Error:
        delimiter_counts = {
            delimiter: sample.count(delimiter)
            for delimiter in [",", ";", "\t", "|"]
        }

        return max(
            delimiter_counts,
            key=delimiter_counts.get
        )


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

DETECTED_DELIMITER = detect_delimiter(
    SOURCE_PATH
)

print("Source SHA-256:", SOURCE_SHA256)
print("Detected delimiter:", repr(DETECTED_DELIMITER))


Source SHA-256: eb46e8e036c08032bbd18d234d3da22442111288e26af32d3eacef592457e788
Detected delimiter: ';'


In [5]:
# ============================================================
# 4. Load and standardise the source dataset
# ============================================================

source_df = pd.read_csv(
    SOURCE_PATH,
    sep=DETECTED_DELIMITER,
    encoding="utf-8-sig"
)

source_df.columns = [
    str(column).strip()
    for column in source_df.columns
]

observed_columns = source_df.columns.tolist()

columns_valid = (
    observed_columns
    == EXPECTED_COLUMNS
)

if not columns_valid:
    raise AssertionError(
        "Unexpected D12 column structure. "
        f"Expected {EXPECTED_COLUMNS}, observed {observed_columns}."
    )


df = source_df.copy()

df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
)

df["Annual CO2 emissions"] = pd.to_numeric(
    df["Annual CO2 emissions"],
    errors="coerce"
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
display(df.head(10))
display(df.tail(10))


Dataset shape: (275, 2)
Columns: ['Year', 'Annual CO2 emissions']



,Year,Annual CO2 emissions
0,1750,9305937
1,1751,9407229
2,1752,9505168
3,1753,9610490
4,1754,9733580
5,1755,9793468
6,1756,9909914
7,1757,10093936
8,1758,10216358
9,1759,10338854


,Year,Annual CO2 emissions
265,2015,35403526000
266,2016,35392830000
267,2017,35974610000
268,2018,36734005000
269,2019,37086570000
270,2020,35158230000
271,2021,36866863000
272,2022,37527773000
273,2023,38094040000
274,2024,38598580000


In [6]:
# ============================================================
# 5. Dataset and column profiling
# ============================================================

column_profile_rows = []

for column in df.columns:
    series = df[column]

    non_null_count = int(
        series.notna().sum()
    )

    null_count = int(
        series.isna().sum()
    )

    unique_count = int(
        series.nunique(dropna=True)
    )

    column_profile_rows.append(
        {
            "Column": column,
            "Data Type": str(series.dtype),
            "Row Count": int(len(series)),
            "Non-null Count": non_null_count,
            "Null Count": null_count,
            "Null Rate": (
                null_count / len(series)
                if len(series)
                else 0.0
            ),
            "Unique Count": unique_count,
            "Minimum": (
                float(series.min())
                if pd.api.types.is_numeric_dtype(series)
                and non_null_count
                else None
            ),
            "Maximum": (
                float(series.max())
                if pd.api.types.is_numeric_dtype(series)
                and non_null_count
                else None
            )
        }
    )


column_profile_df = pd.DataFrame(
    column_profile_rows
)

display(column_profile_df)

print()
df.info()

print()
display(
    df.describe(
        include="all"
    )
)


,Column,Data Type,Row Count,Non-null Count,Null Count,Null Rate,Unique Count,Minimum,Maximum
0,Year,int64,275,275,0,0.0,275,1750.0,2.024000e+03
1,Annual CO2 emissions,int64,275,275,0,0.0,275,9305937.0,3.859858e+10



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Year                  275 non-null    int64
 1   Annual CO2 emissions  275 non-null    int64
dtypes: int64(2)
memory usage: 4.4 KB



,Year,Annual CO2 emissions
count,275.000000,2.750000e+02
mean,1887.000000,6.724087e+09
std,79.529869,1.057361e+10
min,1750.000000,9.305937e+06
25%,1818.500000,5.083162e+07
50%,1887.000000,1.084283e+09
75%,1955.500000,7.685189e+09
max,2024.000000,3.859858e+10


In [7]:
# ============================================================
# 6. Time-series integrity checks
# ============================================================

row_count = int(
    len(df)
)

column_count = int(
    df.shape[1]
)

missing_identifier_rows = int(
    df["Year"].isna().sum()
)

missing_value_rows = int(
    df["Annual CO2 emissions"].isna().sum()
)

exact_duplicate_rows = int(
    df.duplicated().sum()
)

duplicate_year_rows = int(
    df.duplicated(
        subset=["Year"],
        keep=False
    ).sum()
)

minimum_year = int(
    df["Year"].min()
)

maximum_year = int(
    df["Year"].max()
)

observed_years = set(
    df["Year"]
    .dropna()
    .astype(int)
    .tolist()
)

expected_years = set(
    range(
        minimum_year,
        maximum_year + 1
    )
)

missing_years = sorted(
    expected_years
    - observed_years
)

year_sequence_strictly_increasing = bool(
    df["Year"]
    .dropna()
    .is_monotonic_increasing
)

annual_frequency_complete = (
    len(missing_years) == 0
)

numeric_values_non_negative = bool(
    (
        df["Annual CO2 emissions"]
        .dropna()
        >= 0
    ).all()
)

identifier_integrity_valid = (
    missing_identifier_rows == 0
    and duplicate_year_rows == 0
)

value_integrity_valid = (
    missing_value_rows == 0
    and numeric_values_non_negative
)

time_series_integrity_valid = all(
    [
        identifier_integrity_valid,
        value_integrity_valid,
        exact_duplicate_rows == 0,
        annual_frequency_complete,
        year_sequence_strictly_increasing
    ]
)


INTEGRITY_REPORT = {
    "document_id": DOCUMENT_ID,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "row_count": row_count,
    "column_count": column_count,
    "minimum_year": minimum_year,
    "maximum_year": maximum_year,
    "expected_annual_observation_count":
        maximum_year - minimum_year + 1,
    "observed_annual_observation_count":
        len(observed_years),
    "missing_years": missing_years,
    "annual_frequency_complete":
        annual_frequency_complete,
    "year_sequence_strictly_increasing":
        year_sequence_strictly_increasing,
    "missing_identifier_rows":
        missing_identifier_rows,
    "missing_value_rows":
        missing_value_rows,
    "exact_duplicate_rows":
        exact_duplicate_rows,
    "duplicate_year_rows":
        duplicate_year_rows,
    "numeric_values_non_negative":
        numeric_values_non_negative,
    "identifier_integrity_valid":
        identifier_integrity_valid,
    "value_integrity_valid":
        value_integrity_valid,
    "time_series_integrity_valid":
        time_series_integrity_valid
}


print(
    json.dumps(
        INTEGRITY_REPORT,
        ensure_ascii=False,
        indent=2
    )
)

if not time_series_integrity_valid:
    raise AssertionError(
        "D12 failed one or more source integrity checks."
    )


{
  "document_id": "D12",
  "source_file": "D12 - ourworldindataCO2.csv",
  "source_file_sha256": "eb46e8e036c08032bbd18d234d3da22442111288e26af32d3eacef592457e788",
  "row_count": 275,
  "column_count": 2,
  "minimum_year": 1750,
  "maximum_year": 2024,
  "expected_annual_observation_count": 275,
  "observed_annual_observation_count": 275,
  "missing_years": [],
  "annual_frequency_complete": true,
  "year_sequence_strictly_increasing": true,
  "missing_identifier_rows": 0,
  "missing_value_rows": 0,
  "exact_duplicate_rows": 0,
  "duplicate_year_rows": 0,
  "numeric_values_non_negative": true,
  "identifier_integrity_valid": true,
  "value_integrity_valid": true,
  "time_series_integrity_valid": true
}


In [8]:
# ============================================================
# 7. Document-level characterisation
# ============================================================


total_non_null_cells = int(
    df.notna().sum().sum()
)

numeric_non_null_cells = int(
    sum(
        df[column].notna().sum()
        for column in df.columns
        if pd.api.types.is_numeric_dtype(
            df[column]
        )
    )
)

total_non_null_cells = int(
    df.notna().sum().sum()
)

numerical_content_density = (
    numeric_non_null_cells
    / total_non_null_cells
    if total_non_null_cells
    else 0.0
)

DOCUMENT_CHARACTERISATION = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "source_format":
        SOURCE_FORMAT,

    "delimiter":
        DETECTED_DELIMITER,

    "document_class":
        "Structured tabular dataset",

    "record_granularity":
        "One annual observation per row",

    "row_count":
        row_count,

    "column_count":
        column_count,

    "column_names":
        observed_columns,

    "numerical_content_density":
        round(
            numerical_content_density,
            3
        ),

    "temporal_coverage": {
        "minimum_year":
            minimum_year,

        "maximum_year":
            maximum_year,

        "frequency":
            "Annual",

        "continuous":
            bool(
                annual_frequency_complete
            )
    },

    "machine_readable":
        True,

    "text_extractable":
        True,

    "ocr_required":
        False,

    "contains_header_row":
        True,

    "contains_tabular_records":
        True,

    "contains_single_indicator_series":
        True,

    "contains_duplicate_years":
        bool(
            duplicate_year_rows > 0
        ),

    "contains_missing_years":
        bool(
            len(missing_years) > 0
        ),

    "contains_missing_values":
        bool(
            missing_value_rows > 0
        ),

    "year_sequence_strictly_increasing":
        bool(
            year_sequence_strictly_increasing
        ),

    "annual_frequency_complete":
        bool(
            annual_frequency_complete
        ),

    "numeric_values_non_negative":
        bool(
            numeric_values_non_negative
        ),

    "source_unit_explicitly_represented":
        False,

    "fixed_extraction_scope": (
        "Twenty-five predefined annual CO2-emission "
        "observations selected before branch execution, "
        "covering long-run historical milestones, modern "
        "decadal observations and every annual observation "
        "from 2011 to 2024."
    ),

    "excluded_from_reference_scope": (
        "All source years not included in the fixed "
        "25-year reference subset."
    ),

    "notes": (
        "D12 is a natively machine-readable two-column "
        "annual time series. The supplied CSV explicitly "
        "contains Year and Annual CO2 emissions but does "
        "not provide a separate measurement-unit field. "
        "No external unit information is introduced into "
        "the fixed reference dataset."
    ),

}


print(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D12",
  "document_name": "Our World in Data — Annual CO2 emissions time series",
  "source_file": "D12 - ourworldindataCO2.csv",
  "source_file_sha256": "eb46e8e036c08032bbd18d234d3da22442111288e26af32d3eacef592457e788",
  "source_format": "CSV",
  "delimiter": ";",
  "document_class": "Structured tabular dataset",
  "record_granularity": "One annual observation per row",
  "row_count": 275,
  "column_count": 2,
  "column_names": [
    "Year",
    "Annual CO2 emissions"
  ],
  "numerical_content_density": 1.0,
  "temporal_coverage": {
    "minimum_year": 1750,
    "maximum_year": 2024,
    "frequency": "Annual",
    "continuous": true
  },
  "machine_readable": true,
  "text_extractable": true,
  "ocr_required": false,
  "contains_header_row": true,
  "contains_tabular_records": true,
  "contains_single_indicator_series": true,
  "contains_duplicate_years": false,
  "contains_missing_years": false,
  "contains_missing_values": false,
  "year_sequence_strictly_increa

In [9]:
# ============================================================
# 8. Indicator-level document assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Reading Order Quality",

        "Score":
            "Low",

        "Evidence Source":
            "Automated time-series profiling",

        "Justification":
            "Each row represents one annual observation, "
            "years are stored in chronological order, and "
            "no spatial or multi-region reading sequence "
            "must be reconstructed."
    },
    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Table Structure Integrity",

        "Score":
            "Low",

        "Evidence Source":
            "Automated dataset profiling",

        "Justification":
            "All 275 observations follow the same simple "
            "two-column structure consisting of Year and "
            "Annual CO2 emissions."
    },
    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Section/Header Hierarchy",

        "Score":
            "Low",

        "Evidence Source":
            "Header inspection",

        "Justification":
            "The source contains one explicit header row "
            "and requires no nested section or heading "
            "hierarchy to interpret the observations."
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "Sharpness",

        "Score":
            "Low",

        "Evidence Source":
            "File-format inspection",

        "Justification":
            "The CSV is natively machine-readable, so "
            "image sharpness does not constrain information "
            "recovery."
    },
    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "Noise / Degradation",

        "Score":
            "Low",

        "Evidence Source":
            "File-format inspection",

        "Justification":
            "No scanning noise, blur or visual degradation "
            "affects the machine-readable CSV representation."
    },
    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "OCR Dependency",

        "Score":
            "Low",

        "Evidence Source":
            "File-format inspection",

        "Justification":
            "All relevant information is represented "
            "directly as structured CSV data and OCR is "
            "not required."
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Terminology Consistency",

        "Score":
            "Low",

        "Evidence Source":
            "Header and dataset inspection",

        "Justification":
            "The same Year and Annual CO2 emissions labels "
            "apply consistently to all observations."
    },
    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Schema Alignment",

        "Score":
            "Low",

        "Evidence Source":
            "Reference-schema comparison",

        "Justification":
            "The source Year and Annual CO2 emissions "
            "columns map directly to the reporting-period "
            "and value concepts required by the extraction "
            "task."
    },
    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Numerical Density",

        "Score":
            "High",

        "Evidence Source":
            "Automated dataset profiling",

        "Justification":
            "The dataset is almost entirely quantitative, "
            "with every observation containing a numerical "
            "year identifier and a numerical emissions value."
    },

    {
        "Dimension":
            "Completeness and Consistency",

        "Indicator":
            "Required Field Presence",

        "Score":
            "Low",

        "Evidence Source":
            "Automated source and reference verification",

        "Justification":
            "All source observations contain the required "
            "Year and Annual CO2 emissions values, and all "
            "25 predefined reference years are present."
    },
    {
        "Dimension":
            "Completeness and Consistency",

        "Indicator":
            "Internal Consistency",

        "Score":
            "Low",

        "Evidence Source":
            "Automated time-series integrity checks",

        "Justification":
            "Years are unique, continuous and strictly "
            "increasing, with no missing annual observations "
            "or duplicate identifiers."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",

        "Indicator":
            "Format Heterogeneity",

        "Score":
            "Low",

        "Evidence Source":
            "Automated dataset profiling",

        "Justification":
            "The complete document uses one uniform "
            "two-column tabular representation with no "
            "embedded alternative formats."
    },
    {
        "Dimension":
            "Representation and Normalisation Complexity",

        "Indicator":
            "Unit / Label Variability",

        "Score":
            "Low",

        "Evidence Source":
            "Header and source inspection",

        "Justification":
            "Column labels and value representation remain "
            "stable throughout the dataset. The absence of "
            "an explicit source-unit field is consistent "
            "across all observations rather than variable "
            "within the document."
    }
]


indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

display(
    indicator_assessment_df
)

,Dimension,Indicator,Score,Evidence Source,Justification
0,Structural Readiness,Reading Order Quality,Low,Automated time-series profiling,"Each row represents one annual observation, ye..."
1,Structural Readiness,Table Structure Integrity,Low,Automated dataset profiling,All 275 observations follow the same simple tw...
2,Structural Readiness,Section/Header Hierarchy,Low,Header inspection,The source contains one explicit header row an...
3,Visual/OCR Readiness,Sharpness,Low,File-format inspection,"The CSV is natively machine-readable, so image..."
4,Visual/OCR Readiness,Noise / Degradation,Low,File-format inspection,"No scanning noise, blur or visual degradation ..."
5,Visual/OCR Readiness,OCR Dependency,Low,File-format inspection,All relevant information is represented direct...
6,Semantic Quality,Terminology Consistency,Low,Header and dataset inspection,The same Year and Annual CO2 emissions labels ...
7,Semantic Quality,Schema Alignment,Low,Reference-schema comparison,The source Year and Annual CO2 emissions colum...
8,Semantic Quality,Numerical Density,High,Automated dataset profiling,"The dataset is almost entirely quantitative, w..."
9,Completeness and Consistency,Required Field Presence,Low,Automated source and reference verification,All source observations contain the required Y...


In [10]:
# ============================================================
# 9. Validate indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}

expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}


invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)


observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)


missing_indicators = (
    expected_indicators
    - observed_indicators
)


unexpected_indicators = (
    observed_indicators
    - expected_indicators
)


if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores: "
        f"{invalid_scores}"
    )


if missing_indicators:
    raise ValueError(
        f"Missing indicators: "
        f"{missing_indicators}"
    )


if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators: "
        f"{unexpected_indicators}"
    )


if len(
    indicator_assessment_df
) != len(
    expected_indicators
):
    raise ValueError(
        "Duplicate indicator rows detected."
    )


print(
    "Indicator assessment validation passed."
)

Indicator assessment validation passed.


In [11]:
# ============================================================
# 10. Dimension-level assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}


indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(
    score_to_numeric
)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),

        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(
    mean_score
):
    if mean_score < 1.5:
        return "Low"

    elif mean_score < 2.5:
        return "Medium"

    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)


dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(
    2
)


display(
    dimension_assessment_df
)

,Dimension,Mean_Score,Number_of_Indicators,Dimension Score
0,Completeness and Consistency,1.00,2,Low
1,Representation and Normalisation Complexity,1.00,2,Low
2,Semantic Quality,1.67,3,Medium
3,Structural Readiness,1.00,3,Low
4,Visual/OCR Readiness,1.00,3,Low


In [12]:
# ============================================================
# 11. Structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            "Arithmetic mean of indicator scores within "
            "each dimension.",

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}


print(
    json.dumps(
        QUALITY_EVIDENCE,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D12",
  "assessment_basis": "Observed document evidence was mapped to the predefined Low, Medium, and High operational criteria defined in Table 3.3 of the methodology.",
  "evidence_method": "Evidence was obtained through automated profiling where measurable characteristics could be derived programmatically and through documented manual inspection where qualitative assessment was required.",
  "indicators": [
    {
      "Dimension": "Structural Readiness",
      "Indicator": "Reading Order Quality",
      "Score": "Low",
      "Evidence Source": "Automated time-series profiling",
      "Justification": "Each row represents one annual observation, years are stored in chronological order, and no spatial or multi-region reading sequence must be reconstructed."
    },
    {
      "Dimension": "Structural Readiness",
      "Indicator": "Table Structure Integrity",
      "Score": "Low",
      "Evidence Source": "Automated dataset profiling",
      "Justification": "All 

In [13]:
# ============================================================
# 12. Fixed extraction task
# ============================================================

EXTRACTION_TASK = """
Extract the fixed set of annual CO2-emissions observations represented
in the supplied D12 CSV dataset.

Treat the supplied CSV as the only source.

Return exactly 25 records corresponding to the predefined reference
years.

For every record return exactly:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Rules:

- Extract only values explicitly represented in the supplied CSV.
- Preserve the exact year-to-value association.
- Preserve the source numerical value without conversion.
- Do not interpolate, aggregate, round, calculate or derive values.
- Do not infer a measurement unit that is not explicitly represented
  in the supplied CSV.
- Use null for Unit because no separate measurement unit is explicitly
  represented in the supplied source.
- Use the physical CSV row containing the observation as Source Location,
  treating the header as physical row 1.
- Return one record for every predefined reference year.
- Return exactly 25 records.
- Return valid JSON using the exact field names defined in the
  extraction schema.
- Do not include explanations before or after the JSON.
"""


print(
    EXTRACTION_TASK
)


Extract the fixed set of annual CO2-emissions observations represented
in the supplied D12 CSV dataset.

Treat the supplied CSV as the only source.

Return exactly 25 records corresponding to the predefined reference
years.

For every record return exactly:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Rules:

- Extract only values explicitly represented in the supplied CSV.
- Preserve the exact year-to-value association.
- Preserve the source numerical value without conversion.
- Do not interpolate, aggregate, round, calculate or derive values.
- Do not infer a measurement unit that is not explicitly represented
  in the supplied CSV.
- Use null for Unit because no separate measurement unit is explicitly
  represented in the supplied source.
- Use the physical CSV row containing the observation as Source Location,
  treating the header as physical row 1.
- Return one record for every predefined reference year.
- Return exactly 25 records.
- Re

In [14]:
# ============================================================
# 13. Extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "selected_annual_environmental_time_series_observation",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category": {
            "type": [
                "string",
                "null"
            ]
        },

        "Topic": {
            "type": [
                "string",
                "null"
            ]
        },

        "Description": {
            "type": [
                "string",
                "null"
            ]
        },

        "Value": {
            "type": [
                "number",
                "null"
            ]
        },

        "Unit": {
            "type": [
                "string",
                "null"
            ]
        },

        "Reporting Period": {
            "type": [
                "string",
                "null"
            ]
        },

        "Source Location": {
            "type": [
                "string",
                "null"
            ]
        }
    },

    "expected_output_structure": {
        "document_id":
            DOCUMENT_ID,

        "records": [
            {
                "Category":
                    "string or null",

                "Topic":
                    "string or null",

                "Description":
                    "string or null",

                "Value":
                    "number or null",

                "Unit":
                    "string or null",

                "Reporting Period":
                    "string or null",

                "Source Location":
                    "string or null"
            }
        ]
    }
}


print(
    json.dumps(
        EXTRACTION_SCHEMA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D12",
  "record_level": "selected_annual_environmental_time_series_observation",
  "expected_record_count": 25,
  "fields": {
    "Category": {
      "type": [
        "string",
        "null"
      ]
    },
    "Topic": {
      "type": [
        "string",
        "null"
      ]
    },
    "Description": {
      "type": [
        "string",
        "null"
      ]
    },
    "Value": {
      "type": [
        "number",
        "null"
      ]
    },
    "Unit": {
      "type": [
        "string",
        "null"
      ]
    },
    "Reporting Period": {
      "type": [
        "string",
        "null"
      ]
    },
    "Source Location": {
      "type": [
        "string",
        "null"
      ]
    }
  },
  "expected_output_structure": {
    "document_id": "D12",
    "records": [
      {
        "Category": "string or null",
        "Topic": "string or null",
        "Description": "string or null",
        "Value": "number or null",
        "Unit": "string or null",
 

In [15]:
# ============================================================
# 14. Reference schema
# ============================================================

REFERENCE_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "selected_annual_environmental_time_series_observation",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "expected_category_counts": {
        REFERENCE_CATEGORY:
            EXPECTED_REFERENCE_RECORD_COUNT
    },

    "fields": {
        "Category":
            "Fixed reference category.",

        "Topic":
            "Source-grounded indicator represented by the record.",

        "Description":
            "Concise source-grounded description of the observation.",

        "Value":
            "Annual CO2 emissions value explicitly represented in the source CSV.",

        "Unit":
            (
                "Measurement unit explicitly represented in the source, "
                "or null when no source-unit field is available."
            ),

        "Reporting Period":
            "Calendar year associated with the source observation.",

        "Source Location":
            (
                "Physical CSV row containing the observation, "
                "with the header treated as physical row 1."
            )
    },

    "null_policy":
        (
            "Unit is null because the supplied CSV does not explicitly "
            "represent a separate measurement unit."
        ),

    "selection_policy": {
        "historical_milestones":
            [
                1750,
                1800,
                1850,
                1900,
                1950
            ],

        "modern_decadal_observations":
            [
                1960,
                1970,
                1980,
                1990,
                2000,
                2010
            ],

        "recent_annual_observations":
            list(
                range(
                    2011,
                    2025
                )
            )
    },

    "construction_method":
        (
            "Deterministic selection of predefined source rows "
            "followed by programmatic integrity verification."
        ),

    "preservation_rules": [
        "Preserve exact source values.",
        "Preserve exact year-to-value association.",
        "Do not interpolate missing observations.",
        "Do not aggregate source observations.",
        "Do not calculate or derive values.",
        "Do not infer a measurement unit from external information."
    ],

    "branch_reuse":
        (
            "The same fixed reference dataset is reused for "
            "Branches A, B and C."
        )
}


print(
    json.dumps(
        REFERENCE_SCHEMA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D12",
  "record_level": "selected_annual_environmental_time_series_observation",
  "expected_record_count": 25,
  "expected_category_counts": {
    "Environmental time-series": 25
  },
  "fields": {
    "Category": "Fixed reference category.",
    "Topic": "Source-grounded indicator represented by the record.",
    "Description": "Concise source-grounded description of the observation.",
    "Value": "Annual CO2 emissions value explicitly represented in the source CSV.",
    "Unit": "Measurement unit explicitly represented in the source, or null when no source-unit field is available.",
    "Reporting Period": "Calendar year associated with the source observation.",
    "Source Location": "Physical CSV row containing the observation, with the header treated as physical row 1."
  },
  "null_policy": "Unit is null because the supplied CSV does not explicitly represent a separate measurement unit.",
  "selection_policy": {
    "historical_milestones": [
      1750,
   

In [16]:
# ============================================================
# 15. Reference dataset construction
# ============================================================

available_years = set(
    df["Year"]
    .dropna()
    .astype(int)
    .tolist()
)

missing_reference_years = sorted(
    set(REFERENCE_YEARS)
    - available_years
)

print(
    "Missing predefined reference years:",
    missing_reference_years
)

if missing_reference_years:
    raise AssertionError(
        "One or more predefined D12 reference years "
        "are missing from the source dataset."
    )

reference_source_df = (
    df.reset_index()
    .loc[
        lambda frame:
            frame["Year"]
            .astype(int)
            .isin(REFERENCE_YEARS)
    ]
    .copy()
)


reference_source_df["Year"] = (
    reference_source_df["Year"]
    .astype(int)
)


reference_source_df[
    "_reference_order"
] = (
    reference_source_df[
        "Year"
    ]
    .map(
        {
            year: position
            for position, year
            in enumerate(REFERENCE_YEARS)
        }
    )
)


reference_source_df = (
    reference_source_df
    .sort_values(
        "_reference_order"
    )
    .drop(
        columns=[
            "_reference_order"
        ]
    )
    .reset_index(
        drop=True
    )
)


reference_rows = []


for _, row in reference_source_df.iterrows():

    source_index = int(
        row["index"]
    )

    year = int(
        row["Year"]
    )

    source_value = row[
        "Annual CO2 emissions"
    ]

    if pd.isna(
        source_value
    ):
        value = None

    elif float(
        source_value
    ).is_integer():
        value = int(
            source_value
        )

    else:
        value = float(
            source_value
        )


    reference_rows.append(
        {
            "Category":
                REFERENCE_CATEGORY,

            "Topic":
                "Annual CO2 emissions",

            "Description":
                "Annual CO2 emissions",

            "Value":
                value,

            "Unit":
                REFERENCE_UNIT,

            "Reporting Period":
                str(
                    year
                ),

            "Source Location":
                f"CSV data row {source_index + 2}"
        }
    )


reference_values_df = pd.DataFrame(
    reference_rows,
    columns=REFERENCE_FIELDS
)


print(
    "Reference records constructed:",
    len(
        reference_values_df
    )
)

display(
    reference_values_df
)

Missing predefined reference years: []
Reference records constructed: 25


,Category,Topic,Description,Value,Unit,Reporting Period,Source Location
0,Environmental time-series,Annual CO2 emissions,Annual CO2 emissions,9305937,None,1750,CSV data row 2
1,Environmental time-series,Annual CO2 emissions,Annual CO2 emissions,32798352,None,1800,CSV data row 52
2,Environmental time-series,Annual CO2 emissions,Annual CO2 emissions,196847600,None,1850,CSV data row 102
3,Environmental time-series,Annual CO2 emissions,Annual CO2 emissions,1962883800,None,1900,CSV data row 152
4,Environmental time-series,Annual CO2 emissions,Annual CO2 emissions,5930406000,None,1950,CSV data row 202
5,Environmental time-series,Annual CO2 emissions,Annual CO2 emissions,9386952000,None,1960,CSV data row 212
6,Environmental time-series,Annual CO2 emissions,Annual CO2 emissions,14899139000,None,1970,CSV data row 222
7,Environmental time-series,Annual CO2 emissions,Annual CO2 emissions,19409793000,None,1980,CSV data row 232
8,Environmental time-series,Annual CO2 emissions,Annual CO2 emissions,22732145000,None,1990,CSV data row 242
9,Environmental time-series,Annual CO2 emissions,Annual CO2 emissions,25511483000,None,2000,CSV data row 252


In [17]:
# ============================================================
# 16. Reference-value validation
# ============================================================

reference_schema_valid = (
    reference_values_df.columns.tolist()
    == REFERENCE_FIELDS
)


reference_record_count_valid = (
    len(reference_values_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)


observed_category_counts = (
    reference_values_df[
        "Category"
    ]
    .value_counts()
    .to_dict()
)


expected_category_counts = {
    REFERENCE_CATEGORY:
        EXPECTED_REFERENCE_RECORD_COUNT
}


category_counts_valid = (
    observed_category_counts
    == expected_category_counts
)


MANDATORY_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Reporting Period",
    "Source Location"
]


NULLABLE_STRING_FIELDS = [
    "Unit"
]


type_issue_rows = []
missing_mandatory_rows = []


for row_index, row in reference_values_df.iterrows():

    for field in MANDATORY_STRING_FIELDS:

        value = row[field]

        if value is None or value == "":

            missing_mandatory_rows.append(
                {
                    "Record Index":
                        int(row_index),

                    "Field":
                        field
                }
            )

        elif not isinstance(value, str):

            type_issue_rows.append(
                {
                    "Record Index":
                        int(row_index),

                    "Field":
                        field,

                    "Observed Type":
                        type(value).__name__
                }
            )


    for field in NULLABLE_STRING_FIELDS:

        value = row[field]

        if (
            value is not None
            and not isinstance(value, str)
        ):

            type_issue_rows.append(
                {
                    "Record Index":
                        int(row_index),

                    "Field":
                        field,

                    "Observed Type":
                        type(value).__name__
                }
            )


    value = row["Value"]

    if (
        value is None
        or isinstance(value, bool)
        or not isinstance(
            value,
            (int, float)
        )
    ):

        type_issue_rows.append(
            {
                "Record Index":
                    int(row_index),

                "Field":
                    "Value",

                "Observed Type":
                    type(value).__name__
            }
        )


type_issues_df = pd.DataFrame(
    type_issue_rows
)

missing_mandatory_df = pd.DataFrame(
    missing_mandatory_rows
)


field_types_valid = (
    type_issues_df.empty
)

mandatory_fields_complete = (
    missing_mandatory_df.empty
)


reference_duplicate_count = int(
    reference_values_df
    .duplicated()
    .sum()
)


reference_periods_unique = bool(
    reference_values_df[
        "Reporting Period"
    ].nunique()
    == EXPECTED_REFERENCE_RECORD_COUNT
)


reference_values_non_negative = bool(
    (
        reference_values_df[
            "Value"
        ]
        >= 0
    ).all()
)


reference_locations_valid = bool(
    reference_values_df[
        "Source Location"
    ]
    .str.fullmatch(
        r"CSV data row \d+"
    )
    .all()
)


reference_units_null = bool(
    reference_values_df[
        "Unit"
    ].isna().all()
)


observed_reference_years = (
    reference_values_df[
        "Reporting Period"
    ]
    .astype(int)
    .tolist()
)


reference_years_valid = (
    observed_reference_years
    == REFERENCE_YEARS
)


reference_integrity_valid = all(
    [
        reference_schema_valid,
        reference_record_count_valid,
        category_counts_valid,
        field_types_valid,
        mandatory_fields_complete,
        reference_duplicate_count == 0,
        reference_periods_unique,
        reference_values_non_negative,
        reference_locations_valid,
        reference_units_null,
        reference_years_valid
    ]
)


REFERENCE_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "expected_reference_records":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_records":
        int(
            len(reference_values_df)
        ),

    "reference_schema_valid":
        bool(reference_schema_valid),

    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "expected_category_counts":
        expected_category_counts,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        bool(category_counts_valid),

    "field_types_valid":
        bool(field_types_valid),

    "mandatory_fields_complete":
        bool(mandatory_fields_complete),

    "reference_duplicate_count":
        int(reference_duplicate_count),

    "reference_periods_unique":
        bool(reference_periods_unique),

    "reference_values_non_negative":
        bool(reference_values_non_negative),

    "reference_locations_valid":
        bool(reference_locations_valid),

    "reference_units_null":
        bool(reference_units_null),

    "reference_years_valid":
        bool(reference_years_valid),

    "reference_integrity_passed":
        bool(reference_integrity_valid)
}


print(
    json.dumps(
        REFERENCE_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)


if not field_types_valid:
    display(type_issues_df)


if not mandatory_fields_complete:
    display(missing_mandatory_df)


if not reference_integrity_valid:
    raise AssertionError(
        "D12 reference-value validation failed."
    )

{
  "document_id": "D12",
  "expected_reference_records": 25,
  "observed_reference_records": 25,
  "reference_schema_valid": true,
  "reference_record_count_valid": true,
  "expected_category_counts": {
    "Environmental time-series": 25
  },
  "observed_category_counts": {
    "Environmental time-series": 25
  },
  "category_counts_valid": true,
  "field_types_valid": true,
  "mandatory_fields_complete": true,
  "reference_duplicate_count": 0,
  "reference_periods_unique": true,
  "reference_values_non_negative": true,
  "reference_locations_valid": true,
  "reference_units_null": true,
  "reference_years_valid": true,
  "reference_integrity_passed": true
}


In [18]:
# ============================================================
# 17. Dataset statistics and reference metadata
# ============================================================

DATASET_STATISTICS = {
    "document_id":
        DOCUMENT_ID,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "rows":
        row_count,

    "columns":
        column_count,

    "column_names":
        observed_columns,

    "data_types": {
        column:
            str(
                dtype
            )

        for column, dtype
        in df.dtypes.items()
    },

    "minimum_year":
        minimum_year,

    "maximum_year":
        maximum_year,

    "year_count":
        len(
            observed_years
        ),

    "minimum_annual_co2_emissions":
        float(
            df[
                "Annual CO2 emissions"
            ].min()
        ),

    "maximum_annual_co2_emissions":
        float(
            df[
                "Annual CO2 emissions"
            ].max()
        ),

    "null_count_by_column": {
        column:
            int(
                count
            )

        for column, count
        in df.isna().sum().items()
    },

    "unique_count_by_column": {
        column:
            int(
                count
            )

        for column, count
        in df.nunique(
            dropna=True
        ).items()
    }
}


REFERENCE_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "reference_record_count":
        int(
            len(
                reference_values_df
            )
        ),

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "record_count_valid":
        bool(
            reference_record_count_valid
        ),

    "category_counts":
        observed_category_counts,

    "reference_years":
        REFERENCE_YEARS,

    "minimum_reference_year":
        min(
            REFERENCE_YEARS
        ),

    "maximum_reference_year":
        max(
            REFERENCE_YEARS
        ),

    "unit_explicitly_present_in_source":
        False,

    "unit_stored_as_null":
        bool(
            reference_units_null
        ),

    "reference_values_non_negative":
        bool(
            reference_values_non_negative
        ),

    "reference_integrity_passed":
        bool(
            reference_integrity_valid
        ),

    "reference_values_branch_independent":
        True,

    "reference_values_reused_across_branches":
        True
}


REFERENCE_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "source_format":
        SOURCE_FORMAT,

    "reference_file":
        REFERENCE_VALUES_PATH.name,

    "reference_construction_method":
        (
            "Deterministic source-row selection "
            "using predefined reference years."
        ),

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_record_count":
        int(
            len(
                reference_values_df
            )
        ),

    "reference_fields":
        REFERENCE_FIELDS,

    "reference_years":
        REFERENCE_YEARS,

    "selection_policy":
        (
            "Five long-run historical milestones, "
            "six modern decadal observations and "
            "every annual observation from 2011 "
            "through 2024."
        ),

    "selection_reason":
        (
            "The fixed 25-record subset preserves "
            "long-run and recent temporal coverage "
            "while keeping the extraction output "
            "manageable and directly comparable "
            "across Branches A, B and C."
        ),

    "source_unit_explicitly_represented":
        False,

    "unit_inference_applied":
        False,

    "values_calculated":
        False,

    "values_aggregated":
        False,

    "values_interpolated":
        False,

    "manual_correction_applied":
        False,

    "semantic_inference_applied":
        False,

    "reference_values_branch_independent":
        True,

    "reference_values_to_be_reused_for_branches": [
        "A",
        "B",
        "C"
    ],

    "notes":
        (
            "The supplied D12 CSV does not contain a "
            "separate measurement-unit field. Unit is "
            "therefore retained as null rather than "
            "being inferred from external information."
        )
}


print(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

print(
    json.dumps(
        REFERENCE_METADATA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D12",
  "document_name": "Our World in Data — Annual CO2 emissions time series",
  "reference_record_count": 25,
  "expected_record_count": 25,
  "record_count_valid": true,
  "category_counts": {
    "Environmental time-series": 25
  },
  "reference_years": [
    1750,
    1800,
    1850,
    1900,
    1950,
    1960,
    1970,
    1980,
    1990,
    2000,
    2010,
    2011,
    2012,
    2013,
    2014,
    2015,
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
  ],
  "minimum_reference_year": 1750,
  "maximum_reference_year": 2024,
  "unit_explicitly_present_in_source": false,
  "unit_stored_as_null": true,
  "reference_values_non_negative": true,
  "reference_integrity_passed": true,
  "reference_values_branch_independent": true,
  "reference_values_reused_across_branches": true
}
{
  "document_id": "D12",
  "document_name": "Our World in Data — Annual CO2 emissions time series",
  "source_file": "D12 - ourworldindataCO2

In [19]:
# ============================================================
# 18. Export Stage 1 artefacts
# ============================================================

reference_values_df.to_csv(
    REFERENCE_VALUES_PATH,
    index=False,
    encoding="utf-8-sig"
)


REFERENCE_VALUES_JSON_PATH.write_text(
    json.dumps(
        reference_values_df
        .astype(
            object
        )
        .where(
            pd.notna(
                reference_values_df
            ),
            None
        )
        .to_dict(
            orient="records"
        ),
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)


dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)


column_profile_df.to_csv(
    COLUMN_PROFILE_PATH,
    index=False,
    encoding="utf-8-sig"
)


EXTRACTION_TASK_PATH.write_text(
    EXTRACTION_TASK.strip(),
    encoding="utf-8",
    newline="\n"
)


json_outputs = [
    (
        DOCUMENT_CHARACTERISATION_PATH,
        DOCUMENT_CHARACTERISATION
    ),
    (
        DATASET_STATISTICS_PATH,
        DATASET_STATISTICS
    ),
    (
        INTEGRITY_REPORT_PATH,
        INTEGRITY_REPORT
    ),
    (
        EXTRACTION_SCHEMA_PATH,
        EXTRACTION_SCHEMA
    ),
    (
        REFERENCE_SCHEMA_PATH,
        REFERENCE_SCHEMA
    ),
    (
        REFERENCE_SUMMARY_PATH,
        REFERENCE_SUMMARY
    ),
    (
        REFERENCE_METADATA_PATH,
        REFERENCE_METADATA
    ),
    (
        REFERENCE_INTEGRITY_PATH,
        REFERENCE_INTEGRITY
    ),
    (
        QUALITY_EVIDENCE_PATH,
        QUALITY_EVIDENCE
    )
]


for output_path, content in json_outputs:

    with output_path.open(
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            content,
            file,
            ensure_ascii=False,
            indent=2
        )


print(
    "D12 Stage 1 outputs exported successfully."
)

D12 Stage 1 outputs exported successfully.


In [20]:
# ============================================================
# 19. Final Stage 1 checks
# ============================================================

GENERATED_OUTPUTS = [
    DOCUMENT_CHARACTERISATION_PATH,
    DATASET_STATISTICS_PATH,
    INTEGRITY_REPORT_PATH,

    REFERENCE_VALUES_PATH,
    REFERENCE_VALUES_JSON_PATH,

    EXTRACTION_SCHEMA_PATH,
    EXTRACTION_TASK_PATH,

    REFERENCE_SCHEMA_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_METADATA_PATH,
    REFERENCE_INTEGRITY_PATH,

    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    QUALITY_EVIDENCE_PATH,

    COLUMN_PROFILE_PATH
]


missing_outputs = [
    path.name
    for path in GENERATED_OUTPUTS
    if not path.exists()
]


all_outputs_exist = (
    len(
        missing_outputs
    )
    == 0
)


FINAL_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "source_rows":
        row_count,

    "source_columns":
        column_count,

    "source_integrity_valid":
        bool(
            time_series_integrity_valid
        ),

    "reference_records":
        int(
            len(
                reference_values_df
            )
        ),

    "expected_reference_records":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "reference_integrity_passed":
        bool(
            reference_integrity_valid
        ),

    "reference_year_range": {
        "minimum":
            min(
                REFERENCE_YEARS
            ),

        "maximum":
            max(
                REFERENCE_YEARS
            )
    },

    "all_outputs_exist":
        bool(
            all_outputs_exist
        ),

    "outputs_created": [
        path.name
        for path in GENERATED_OUTPUTS
    ],

    "status":
        (
            "Completed successfully"
            if all_outputs_exist
            else "Incomplete"
        )
}


print(
    json.dumps(
        FINAL_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


if not time_series_integrity_valid:
    raise AssertionError(
        "D12 source time-series integrity failed."
    )


if not reference_integrity_valid:
    raise AssertionError(
        "D12 reference-value integrity failed."
    )


if not all_outputs_exist:
    raise AssertionError(
        f"Missing D12 output files: "
        f"{missing_outputs}"
    )


print(
    "\nD12 Stage 1 completed successfully."
)

{
  "document_id": "D12",
  "source_rows": 275,
  "source_columns": 2,
  "source_integrity_valid": true,
  "reference_records": 25,
  "expected_reference_records": 25,
  "reference_integrity_passed": true,
  "reference_year_range": {
    "minimum": 1750,
    "maximum": 2024
  },
  "all_outputs_exist": true,
  "outputs_created": [
    "D12_document_characterisation.json",
    "D12_dataset_statistics.json",
    "D12_integrity_report.json",
    "D12_reference_values.csv",
    "D12_reference_values.json",
    "D12_extraction_schema.json",
    "D12_extraction_task.txt",
    "D12_reference_schema.json",
    "D12_reference_summary.json",
    "D12_reference_metadata.json",
    "D12_reference_integrity.json",
    "D12_indicator_assessment.csv",
    "D12_dimension_assessment.csv",
    "D12_quality_evidence.json",
    "D12_column_profile.csv"
  ],
  "status": "Completed successfully"
}

D12 Stage 1 completed successfully.


In [21]:
# ============================================================
# 20. List generated outputs
# ============================================================

print(
    "Generated D12 Stage 1 files:\n"
)


for path in GENERATED_OUTPUTS:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )

Generated D12 Stage 1 files:

- D12_document_characterisation.json | exists: True
- D12_dataset_statistics.json | exists: True
- D12_integrity_report.json | exists: True
- D12_reference_values.csv | exists: True
- D12_reference_values.json | exists: True
- D12_extraction_schema.json | exists: True
- D12_extraction_task.txt | exists: True
- D12_reference_schema.json | exists: True
- D12_reference_summary.json | exists: True
- D12_reference_metadata.json | exists: True
- D12_reference_integrity.json | exists: True
- D12_indicator_assessment.csv | exists: True
- D12_dimension_assessment.csv | exists: True
- D12_quality_evidence.json | exists: True
- D12_column_profile.csv | exists: True
